# TN3 — Hàm loss lai, DS-TCN 192 kênh tầm nhìn 121

## Chỗ lệch mà thí nghiệm này khép lại

Cả đồ án train bằng **MSE** nhưng chấm bằng **Pearson**:

| | dùng gì |
|---|---|
| huấn luyện | **MSE** — phạt sai lệch từng giá trị |
| chọn kênh (`scoring.py:105`) | **Pearson** giữa dự báo và 25 mẫu thật |
| điểm cuối (`scoring.py:214`) | **Pearson** giữa sóng chọn và nhịp thở thật |

Pearson **bất biến với thang đo** — chỉ quan tâm hình dạng. MSE thì ngược lại.
Hai tiêu chí xếp hạng khác nhau:

    dự báo A: đúng hình dạng, biên độ gấp 3   MSE 2,0000   Pearson +1,0000
    dự báo B: phẳng lì, đoán bừa số 0         MSE 0,5000   Pearson  0,0000

MSE chọn B, Pearson chọn A. Model train bằng MSE được thưởng khi **đoán an toàn
về mức trung bình**; model chấm bằng Pearson cần **bắt đúng hình dạng**.

TN3 khép chỗ lệch đó lại:

```
loss = alpha × MSE + (1 − alpha) × (1 − Pearson)
```

`alpha = 1,0` là MSE thuần, tức đúng cấu hình đã chạy.

## Bằng chứng có sẵn

Một khảo sát loss trên cùng dữ liệu này, chạy bằng bộ mã khác, đo trên `val_KL`:

| loss | val_KL |
|---|---:|
| MSE 0,7 + Pearson 0,3 | **0,8657** ± 0,0050 |
| MSE 0,5 + Pearson 0,5 | 0,8603 |
| Pearson thuần | 0,8586 |
| **MSE thuần** | **0,8337** ± 0,0028 |

**+0,032 mà không đụng kiến trúc.** Lớn hơn toàn bộ khoảng cách giữa chín kiến
trúc TN1 (0,0172). Đường cong có hình dạng hợp lý — tăng rồi giảm, đỉnh ở 0,7.

Đó là bộ mã khác, giao thức khác, nên **không dùng làm kết quả**. Nó chỉ nói
đây là hướng đáng thử và vùng alpha nào đáng quét.

## Cấu hình nền

| | |
|---|---|
| model | `ds_tcn --channels 192 --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element` |
| tham số | **310.873** |
| tầm nhìn | 121 |
| MSE thuần | **0,764428** *(1 seed)* |

Chọn cấu hình này vì nó nằm trong nhóm bốn cấu hình đứng đầu đồ án sau TN2.

## Chạy gì

Ba mức alpha, **đủ bốn fold**, một seed:

| alpha | MSE | Pearson |
|---:|---:|---:|
| 0,7 | 70% | 30% |
| 0,5 | 50% | 50% |
| 0,3 | 30% | 70% |

`alpha = 1,0` **không chạy lại** — đó chính là kết quả MSE thuần ở trên.

Mỗi cấu hình khoảng **8 phút mỗi fold**, ba cấu hình khoảng **2,5 giờ**.

## Một lỗi vừa phải sửa trước khi chạy được

`config_id` trước đây **không ghi `alpha`**, nên `--alpha 0.3`, `0.5`, `0.7` ra
cùng một tên. Chạy TN3 mà không sửa thì tệ hơn cả ghi đè: cơ chế bỏ qua fold đã
xong sẽ nuốt luôn hai lần chạy sau, và bảng in ra **ba dòng giống hệt nhau** —
trông như đã chạy đủ.

Giờ tên có dạng `..._mse_pearson_a0.7_corr0.9_seed0`. Hậu tố `_a%g` chỉ xuất
hiện khi loss là `mse_pearson`, nên 16 tên cũ giữ nguyên từng ký tự.

## Đọc kết quả thế nào

So với **0,764428** của chính cấu hình này, cùng bốn fold.

Nhưng nhớ: nền có 1 seed còn ba dòng mới chỉ **một seed**. `seed_std` của tám cấu
hình TN1 trải từ 0,0007 tới 0,0108. Chênh lệch dưới khoảng 0,01 chưa đọc được
gì; chênh lớn hơn thì đáng chạy tiếp hai seed.

## 1. Chuẩn bị Colab

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Tải mã nguồn. Hậu tố `alpha` trong tên cấu hình chỉ có ở bản mới.

In [ ]:
!rm -rf /content/UWB_RADAR
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

Lấy `by_user/` và `windows/` từ Drive.

In [ ]:
!python scripts/restore_processed_data_on_drive.py

Khôi phục kết quả `tn3` đã có.

Ô này gộp `runs/*/summary.csv` vào `runs/summary.csv` — chỗ `run_cv.py` thật sự
tra để biết fold nào đã xong. Tệp nén chỉ chứa bản trong thư mục thí nghiệm,
không chứa bảng chung, nên thiếu bước gộp thì fold cũ bị train lại.

In [ ]:
import glob, subprocess, os, csv
for f in sorted(glob.glob("/content/drive/MyDrive/mobivital/tn3_*c192*.zip")):
    subprocess.run(["unzip", "-oq", f, "-d", "runs/"], check=True)
# run_cv.py tra runs/summary.csv, mà tệp nén chỉ có runs/<thí nghiệm>/summary.csv
rows, seen = [], set()
for s in glob.glob("runs/*/summary.csv"):
    for r in csv.DictReader(open(s)):
        k = (r.get("experiment"), r.get("run_id"))
        if k not in seen: seen.add(k); rows.append(r)
if rows:
    w = csv.DictWriter(open("runs/summary.csv", "w", newline=""), fieldnames=rows[0].keys())
    w.writeheader(); w.writerows(rows)
print("khôi phục", len(rows), "dòng vào runs/summary.csv")

## 2. Kiểm bản cài đặt

Số tham số phải ra đúng **310.873**.

In [ ]:
!python scripts/check_model.py --model ds_tcn --channels 192 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

## 3. Ba mức alpha, đủ 4 fold

Tên cấu hình: `ds_tcn_c192_k5_n4_none_do0.2_dpel_mse_pearson_a<A>_corr0.9_seed0`.

Đọc dòng `cấu hình` ở đầu mỗi lệnh để chắc `alpha` đã vào tên.

**alpha 0,7 — MSE 70%, Pearson 30%**

In [ ]:
!python scripts/run_cv.py --experiment tn3 --model ds_tcn --channels 192 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.7 --seed 0

**alpha 0,5 — MSE 50%, Pearson 50%**

In [ ]:
!python scripts/run_cv.py --experiment tn3 --model ds_tcn --channels 192 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.5 --seed 0

**alpha 0,3 — MSE 30%, Pearson 70%**

In [ ]:
!python scripts/run_cv.py --experiment tn3 --model ds_tcn --channels 192 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.3 --seed 0

## 4. Cất kết quả

In [ ]:
!python scripts/save_results.py tn3 --out tn3_ds_tcn_c192

## 5. Bảng so

Bảng gồm ba mức alpha vừa chạy. Mốc MSE thuần là **0,764428**, nằm ở nhóm `tn1` chứ
không ở `tn3`, nên phải đối chiếu bằng mắt.

In [ ]:
!python scripts/compare_cv.py --experiment tn3

## 6. Đường cong alpha

Trục ngang là trọng số MSE. `alpha = 1,0` là điểm MSE thuần, lấy từ kết quả đã
có chứ không chạy lại.

In [ ]:
import csv, re, matplotlib.pyplot as plt
d = {float(re.search(r"_a([\d.]+)_", r["run_id"]).group(1)): float(r["score_macro"])
     for r in csv.DictReader(open("runs/tn3/summary.csv")) if r["fold"] == "TONG"}
d[1.0] = 0.764428
x = sorted(d); plt.plot(x, [d[i] for i in x], "o-")
plt.xlabel("alpha (trọng số MSE)"); plt.ylabel("cv_score"); plt.grid(alpha=.3)
plt.title("alpha 1,0 = MSE thuần")

## 7. Ngắt phiên

In [ ]:
from google.colab import runtime
runtime.unassign()